# Insurance Linear Regression

This notebook performs linear regression on the insurance dataset. It also applies LabelEncoder and OneHotEncoder to categorical data.

In [ ]:
# Import the relevant statistical and machine learning libraries.
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Load the insuranceData.csv file.
insurance_data = pd.read_csv("insuranceData.csv")

# Display the first few rows to check if the file loaded correctly.
insurance_data.head()

In [ ]:
# Check the column names and data types.
# This helps identify which columns are categorical and need encoding.
insurance_data.info()

In [ ]:
# Apply LabelEncoder to the smoker column.
# This converts the smoker values from yes/no into numeric values.
label_encoder = LabelEncoder()
insurance_data["smokerLabel"] = label_encoder.fit_transform(insurance_data["smoker"])

insurance_data.head()

In [ ]:
# Apply OneHotEncoder to the smoker column.
# This creates separate numeric columns for each category in smoker.

try:
    onehot_encoder = OneHotEncoder(sparse_output=False, drop=None)
except TypeError:
    onehot_encoder = OneHotEncoder(sparse=False, drop=None)

encoded_array = onehot_encoder.fit_transform(insurance_data[["smoker"]])
encoded_columns = onehot_encoder.get_feature_names_out(["smoker"])

encoded_data = pd.DataFrame(
    encoded_array,
    columns=encoded_columns,
    index=insurance_data.index
)

# Combine the encoded columns with the original dataset.
insurance_data_encoded = pd.concat([insurance_data, encoded_data], axis=1)

insurance_data_encoded.head()

In [ ]:
# Select the independent variables/features and the target variable.
# I used age, bmi, children, and smokerLabel to predict medical insurance charges.

X = insurance_data_encoded[["age", "bmi", "children", "smokerLabel"]]
y = insurance_data_encoded["charges"]

In [ ]:
# Split the data into training and testing sets.
# The assignment requires the test size to be 15%.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
# Create and train the Linear Regression model.
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# Use the trained model to make predictions on the test data.
y_pred = model.predict(X_test)

# Evaluate the model using common regression metrics.
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("Root Mean Squared Error:", rmse)
print("R2 Score:", r2)
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

In [ ]:
# Save the predictions back into the dataset.
# I added predictedCharges, residual, and dataSplit columns for the final result CSV file.

insurance_data_encoded["predictedCharges"] = model.predict(X)
insurance_data_encoded["residual"] = insurance_data_encoded["charges"] - insurance_data_encoded["predictedCharges"]
insurance_data_encoded["dataSplit"] = "train"
insurance_data_encoded.loc[X_test.index, "dataSplit"] = "test"

insurance_data_encoded.head()

In [ ]:
# Save the final encoded and predicted data to a CSV file.
insurance_data_encoded.to_csv("insuranceData_result.csv", index=False)

print("Saved file: insuranceData_result.csv")

## Short Conclusion

The model used insurance information such as age, BMI, children, and smoker status to predict medical charges. I encoded the categorical smoker column using both LabelEncoder and OneHotEncoder, then trained a Linear Regression model using 85% of the data and tested it on 15% of the data.